In [9]:
import pandas as pd
import numpy as np
import re
import html
import time
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
possible_paths = [Path("../data/processed/train.csv"), Path("data/processed/train.csv"), Path("./data/processed/train.csv")]
DATA_PATH = next((path for path in possible_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("train.csv not found.")
train_df = pd.read_csv(DATA_PATH)
X = train_df["review"]
y = train_df["sentiment"]
X_train_ablation, X_val_ablation, y_train_ablation, y_val_ablation = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print("Training samples:", len(X_train_ablation))
print("Validation samples:", len(X_val_ablation))

Training samples: 32000
Validation samples: 8000


In [11]:
def remove_html(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)
    return re.sub(r"\s+", " ", text).strip()
def lowercase(text):
    return text.lower()
def remove_punctuation(text):
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()
sample = X.iloc[0]
print("ORIGINAL:")
print(sample[:500])
print("\nHTML REMOVED:")
print(remove_html(sample)[:500])
print("\nLOWERCASED:")
print(lowercase(remove_html(sample))[:500])
print("\nPUNCTUATION REMOVED:")
print(remove_punctuation(lowercase(remove_html(sample)))[:500])

ORIGINAL:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ

HTML REMOVED:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me. The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word. It is calle

In [12]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
stop_words = set(ENGLISH_STOP_WORDS)
def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)
print("Stopword removal function created successfully!")

Stopword removal function created successfully!


In [13]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
for resource in ["wordnet", "omw-1.4"]:
    try:
        nltk.download(resource, quiet=False)
    except Exception as e:
        print(f"Download issue for {resource}: {e}")
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
print("Stem test:", stemmer.stem("playing"))
print("Lemmatization test:", lemmatizer.lemmatize("movies"))
print("NLTK tools loaded successfully!")


Stem test: play
Lemmatization test: movie
NLTK tools loaded successfully!


[nltk_data] Downloading package wordnet to C:\Users\VRINDA
[nltk_data]     ARORA\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\VRINDA
[nltk_data]     ARORA\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [15]:
def stem_text(text):
    return " ".join(stemmer.stem(word) for word in text.split())
def lemmatize_text(text):
    return " ".join(lemmatizer.lemmatize(word) for word in text.split())
print("Stemming and lemmatization functions created successfully!")
def preprocess_e1(text):
    return remove_html(text)
def preprocess_e2(text):
    return lowercase(remove_html(text))
def preprocess_e3(text):
    return remove_punctuation(lowercase(remove_html(text)))
def preprocess_e4(text):
    return remove_stopwords(remove_punctuation(lowercase(remove_html(text))))
def preprocess_e5(text):
    return stem_text(remove_stopwords(remove_punctuation(lowercase(remove_html(text)))))
def preprocess_e6(text):
    return lemmatize_text(remove_stopwords(remove_punctuation(lowercase(remove_html(text)))))
print("All six preprocessing configurations created successfully!")
print("E1:", preprocess_e1(sample)[:300])
print("\nE2:", preprocess_e2(sample)[:300])
print("\nE3:", preprocess_e3(sample)[:300])
print("\nE4:", preprocess_e4(sample)[:300])
print("\nE5:", preprocess_e5(sample)[:300])
print("\nE6:", preprocess_e6(sample)[:300])

Stemming and lemmatization functions created successfully!
All six preprocessing configurations created successfully!
E1: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me. The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this

E2: one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me. the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this

E3: one of the other reviewers has mentioned that after watching just 1 oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right fr

In [ ]:
def run_experiment(name, preprocessing_function):
    print("\n" + "=" * 70)
    print("Running", name)
    print("=" * 70)
    start_time = time.time()
    model = Pipeline([("vectorizer", CountVectorizer(preprocessor=preprocessing_function, lowercase=False, token_pattern=r"(?u)\b\w\w+\b")), ("classifier", LogisticRegression(max_iter=1000, random_state=42))])
    model.fit(X_train_ablation, y_train_ablation)
    y_pred = model.predict(X_val_ablation)
    positive_index = list(model.classes_).index("positive")
    y_prob = model.predict_proba(X_val_ablation)[:, positive_index]
    y_val_binary = (y_val_ablation == "positive").astype(int)
    result = {
        "Experiment": name,
        "Accuracy": accuracy_score(y_val_ablation, y_pred),
        "Precision": precision_score(y_val_ablation, y_pred, pos_label="positive", zero_division=0),
        "Recall": recall_score(y_val_ablation, y_pred, pos_label="positive", zero_division=0),
        "F1-score": f1_score(y_val_ablation, y_pred, pos_label="positive", zero_division=0),
        "ROC-AUC": roc_auc_score(y_val_binary, y_prob),
        "Runtime (seconds)": time.time() - start_time
    }
    for key in ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]:
        print(f"{key}: {result[key]:.4f}")
    print(f"Runtime: {result['Runtime (seconds)']:.2f} seconds")
    return result
print("Experiment function created successfully!")
results = []
results.append(run_experiment("E1 - HTML removal only", preprocess_e1))
results.append(run_experiment("E2 - HTML + lowercase", preprocess_e2))
results.append(run_experiment("E3 - E2 + punctuation removal", preprocess_e3))
results.append(run_experiment("E4 - E3 + stopword removal", preprocess_e4))
results.append(run_experiment("E5 - E4 + stemming", preprocess_e5))
results.append(run_experiment("E6 - E4 + lemmatization", preprocess_e6))
results_df = pd.DataFrame(results)
print("\nPREPROCESSING ABLATION RESULTS")
print("=" * 100)
display(results_df.round(4))

Experiment function created successfully!

Running E1 - HTML removal only
Accuracy: 0.8826
Precision: 0.8787
Recall: 0.8878
F1-score: 0.8832
ROC-AUC: 0.9522
Runtime: 26.37 seconds

Running E2 - HTML + lowercase
Accuracy: 0.8855
Precision: 0.8792
Recall: 0.8938
F1-score: 0.8864
ROC-AUC: 0.9515
Runtime: 29.72 seconds

Running E3 - E2 + punctuation removal
Accuracy: 0.8855
Precision: 0.8792
Recall: 0.8938
F1-score: 0.8864
ROC-AUC: 0.9515
Runtime: 33.37 seconds

Running E4 - E3 + stopword removal
Accuracy: 0.8814
Precision: 0.8747
Recall: 0.8902
F1-score: 0.8824
ROC-AUC: 0.9500
Runtime: 18.57 seconds

Running E5 - E4 + stemming
Accuracy: 0.8810
Precision: 0.8744
Recall: 0.8898
F1-score: 0.8820
ROC-AUC: 0.9452
Runtime: 87.44 seconds

Running E6 - E4 + lemmatization
Accuracy: 0.8808
Precision: 0.8738
Recall: 0.8900
F1-score: 0.8818
ROC-AUC: 0.9488
Runtime: 40.87 seconds

PREPROCESSING ABLATION RESULTS


,Experiment,Accuracy,Precision,Recall,F1-score,ROC-AUC,Runtime (seconds)
0,E1 - HTML removal only,0.8826,0.8787,0.8878,0.8832,0.9522,26.3707
1,E2 - HTML + lowercase,0.8855,0.8792,0.8938,0.8864,0.9515,29.7226
2,E3 - E2 + punctuation removal,0.8855,0.8792,0.8938,0.8864,0.9515,33.3688
3,E4 - E3 + stopword removal,0.8814,0.8747,0.8902,0.8824,0.9500,18.5734
4,E5 - E4 + stemming,0.8810,0.8744,0.8898,0.8820,0.9452,87.4385
5,E6 - E4 + lemmatization,0.8808,0.8738,0.8900,0.8818,0.9488,40.8713


In [ ]:
best_accuracy = results_df.loc[results_df["Accuracy"].idxmax()]
best_f1 = results_df.loc[results_df["F1-score"].idxmax()]
print("Best experiment by Accuracy:")
print(best_accuracy[["Experiment", "Accuracy"]])
print("\nBest experiment by F1-score:")
print(best_f1[["Experiment", "F1-score"]])
comparison_df = results_df.copy()
comparison_df["Accuracy Change vs E1"] = comparison_df["Accuracy"] - comparison_df.loc[0, "Accuracy"]
comparison_df["F1 Change vs E1"] = comparison_df["F1-score"] - comparison_df.loc[0, "F1-score"]
display(comparison_df[["Experiment", "Accuracy", "Accuracy Change vs E1", "F1-score", "F1 Change vs E1", "ROC-AUC", "Runtime (seconds)"]].round(4))

Best experiment by Accuracy:
Experiment    E2 - HTML + lowercase
Accuracy                     0.8855
Name: 1, dtype: object

Best experiment by F1-score:
Experiment    E2 - HTML + lowercase
F1-score                   0.886437
Name: 1, dtype: object


,Experiment,Accuracy,Accuracy Change vs E1,F1-score,F1 Change vs E1,ROC-AUC,Runtime (seconds)
0,E1 - HTML removal only,0.8826,0.0000,0.8832,0.0000,0.9522,26.3707
1,E2 - HTML + lowercase,0.8855,0.0029,0.8864,0.0032,0.9515,29.7226
2,E3 - E2 + punctuation removal,0.8855,0.0029,0.8864,0.0032,0.9515,33.3688
3,E4 - E3 + stopword removal,0.8814,-0.0012,0.8824,-0.0008,0.9500,18.5734
4,E5 - E4 + stemming,0.8810,-0.0016,0.8820,-0.0012,0.9452,87.4385
5,E6 - E4 + lemmatization,0.8808,-0.0019,0.8818,-0.0014,0.9488,40.8713


In [18]:
best_experiment = results_df.loc[results_df["F1-score"].idxmax(), "Experiment"]
best_score = results_df["F1-score"].max()
print("ABLATION STUDY SUMMARY")
print("=" * 70)
print("Best preprocessing configuration:", best_experiment)
print(f"Best validation F1-score: {best_score:.4f}")
if best_experiment.startswith("E1"):
    print("Result: More extensive preprocessing did not improve the baseline F1-score.")
else:
    print("Result: At least one additional preprocessing step improved the baseline F1-score.")
print("\nUse the measured results above when writing the final research conclusion.")

ABLATION STUDY SUMMARY
Best preprocessing configuration: E2 - HTML + lowercase
Best validation F1-score: 0.8864
Result: At least one additional preprocessing step improved the baseline F1-score.

Use the measured results above when writing the final research conclusion.
